# SAAM Project — Part I, Section 2.1: Investment Set & Moment Estimation
**Group AT** — North America (AMER) / Scope 1+2

This notebook loads the cleaned data from Section 1 and, for each allocation year $Y$ (2013–2024):
1. Retrieves the investment set (firms satisfying all eligibility criteria)
2. Extracts the 10-year rolling window of monthly returns ($\tau = 120$ months)
3. Computes the vector of expected returns $\hat{\mu}_Y$ and the covariance matrix $\Sigma_Y$

**Formulas** (from the project):

$$\hat{\mu}_Y = \frac{1}{\tau} \sum_{k=0}^{\tau-1} R_{t-k}$$

$$\Sigma_Y = \frac{1}{\tau} \sum_{k=0}^{\tau-1} (R_{t-k} - \hat{\mu}_Y)(R_{t-k} - \hat{\mu}_Y)'$$

Note: the project uses $1/\tau$ (population estimator), not $1/(\tau-1)$.

## 0. Load Cleaned Data

In [1]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# ── Configuration ──────────────────────────────────────────────────────
ESTIMATION_WINDOW = 120   # τ = 10 years × 12 months
Y0    = 2013              # first allocation year
Y_END = 2024              # last allocation year
DATA_DIR = './cleaned_data/'

# ── Load only what this section needs ─────────────────────────────────
returns_m  = pd.read_csv(f'{DATA_DIR}returns_monthly.csv', index_col=0)
returns_m.columns = pd.to_datetime(returns_m.columns)
firm_names = pd.read_csv(f'{DATA_DIR}firm_names.csv', index_col=0).squeeze()

# Investment sets
inv_df = pd.read_csv(f'{DATA_DIR}investment_sets.csv')
investment_sets = {y: grp['ISIN'].tolist() for y, grp in inv_df.groupby('Year')}

# Date helpers
dates_ret   = returns_m.columns.tolist()
dec_ret_idx = {d.year: i for i, d in enumerate(dates_ret) if d.month == 12}

print(f"Returns: {returns_m.shape[0]} firms × {returns_m.shape[1]} months")
print(f"Date range: {dates_ret[0].strftime('%Y-%m')} → {dates_ret[-1].strftime('%Y-%m')}")
print(f"Investment sets loaded for years: {sorted(investment_sets.keys())}")

Returns: 589 firms × 313 months
Date range: 2000-01 → 2026-01
Investment sets loaded for years: [2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]


## 1. Compute Expected Returns and Covariance Matrix

For each year $Y$, we:
1. Select only the firms in the investment set for that year
2. Extract the $\tau = 120$ monthly returns in the estimation window (ending Dec $Y$)
3. Replace remaining NaN with 0 (missing return = no trade, equivalent to a cash position prior to listing)
4. Compute $\hat{\mu}_Y$ (mean return vector) and $\Sigma_Y$ (covariance matrix) using $1/\tau$

In [2]:
def compute_moments(year, eligible_isins):
    """
    Compute expected return vector and covariance matrix for a given year.
    
    Parameters:
        year: allocation year Y
        eligible_isins: list of ISINs in the investment set
    
    Returns:
        mu_hat: (N,) vector of expected monthly returns
        Sigma:  (N, N) covariance matrix (population estimator, 1/tau)
        window_cols: list of datetime columns used for the estimation
    """
    # Identify the 120-month window ending at Dec of year Y
    end_idx   = dec_ret_idx[year]
    start_idx = end_idx - ESTIMATION_WINDOW + 1
    window_cols = dates_ret[start_idx : end_idx + 1]
    tau = len(window_cols)  # should be 120
    
    # Extract returns for eligible firms in the window
    # NaN → 0: missing returns treated as zero (cash position prior to listing)
    R_arr = returns_m.loc[eligible_isins, window_cols].fillna(0.0).values  # (N, tau)
    
    # Expected return vector: mu_hat = (1/tau) * sum_k R_{t-k}
    mu_hat = R_arr.mean(axis=1)  # (N,)
    
    # Covariance matrix: Sigma = (1/tau) * sum_k (R_{t-k} - mu)(R_{t-k} - mu)'
    R_centered = R_arr - mu_hat[:, np.newaxis]  # (N, tau)
    Sigma = (R_centered @ R_centered.T) / tau    # (N, N)
    
    return mu_hat, Sigma, window_cols

## 2. Run Moment Estimation for All Years

We compute $\hat{\mu}_Y$ and $\Sigma_Y$ for each allocation year and report diagnostics:
- Dimension $N$ and estimation window $\tau$
- Rank of $\Sigma$ (expected: $\tau - 1 = 119$ since centering removes one degree of freedom)
- Condition number on the positive eigenvalues (numerical stability)
- Range of expected returns

In [3]:
# Store results for all years
moments = {}  # {year: (mu_hat, Sigma, eligible_isins, window_cols)}

diag_rows = []
for Y in range(Y0, Y_END + 1):
    eligible = investment_sets[Y]
    N = len(eligible)
    
    mu_hat, Sigma, window_cols = compute_moments(Y, eligible)
    moments[Y] = (mu_hat, Sigma, eligible, window_cols)
    
    # Diagnostics
    eigvals = np.linalg.eigvalsh(Sigma)
    rank = (eigvals > 1e-12).sum()  # numerical rank
    # Condition number: ratio of largest to smallest POSITIVE eigenvalue
    pos_eigvals = eigvals[eigvals > 1e-12]
    cond_num = pos_eigvals[-1] / pos_eigvals[0] if len(pos_eigvals) > 0 else np.inf
    
    diag_rows.append({
        'Year': Y,
        'N': N,
        'τ': len(window_cols),
        'Rank(Σ)': rank,
        'Cond. number': f'{cond_num:.0f}',
        'μ_min (%)': f'{mu_hat.min()*100:.2f}',
        'μ_median (%)': f'{np.median(mu_hat)*100:.2f}',
        'μ_max (%)': f'{mu_hat.max()*100:.2f}',
    })

df_diag = pd.DataFrame(diag_rows).set_index('Year')
df_diag

,N,τ,Rank(Σ),Cond. number,μ_min (%),μ_median (%),μ_max (%)
Year,,,,,,,
2013,337,120,119,1080,-0.49,1.16,3.95
2014,348,120,119,1079,-0.48,1.12,3.24
2015,364,120,119,1115,-1.52,0.96,3.05
2016,381,120,119,1096,-0.53,0.97,3.41
2017,400,120,119,1178,-1.34,1.09,3.17
2018,436,120,119,851,-0.99,1.31,5.31
2019,503,120,119,511,-1.59,1.21,3.42
2020,532,120,119,650,-2.60,1.13,6.68
2021,560,120,119,561,-2.34,1.33,6.29


### Interpretation

Key observations:
- $N > \tau$ for all years (more firms than observations), so $\Sigma_Y$ is **rank-deficient** (rank = $\tau - 1 = 119$). The closed-form formula $\alpha^* = \frac{\Sigma^{-1}e}{e'\Sigma^{-1}e}$ does not work.
- We solve the optimization numerically with the long-only constraint ($\alpha_i \geq 0$), which acts as a natural regularizer.
- Condition numbers are moderate (~1000), meaning the non-zero part of $\Sigma$ is well-conditioned.

## 3. Verification: Covariance Matrix Properties

In [4]:
# Detailed check for the first allocation year
Y_check = 2013
mu_hat, Sigma, eligible, wcols = moments[Y_check]
N = len(eligible)

print(f"Year {Y_check}: N = {N} firms, τ = {len(wcols)} months")

print(f"\nExpected return vector μ_hat:")
print(f"  Shape: ({N},)")
print(f"  Mean:   {mu_hat.mean()*100:.4f}% /month ({mu_hat.mean()*1200:.2f}% /year)")
print(f"  Min:    {mu_hat.min()*100:.4f}% ({firm_names.get(eligible[np.argmin(mu_hat)], '?')})")
print(f"  Max:    {mu_hat.max()*100:.4f}% ({firm_names.get(eligible[np.argmax(mu_hat)], '?')})")

print(f"\nCovariance matrix Σ:")
print(f"  Shape: ({N}, {N})")
print(f"  Symmetric: {np.allclose(Sigma, Sigma.T)}")
eigvals = np.linalg.eigvalsh(Sigma)
print(f"  PSD: {np.all(eigvals >= -1e-10)}  (min eigenvalue: {eigvals[0]:.2e})")
print(f"  Rank: {(eigvals > 1e-12).sum()} / {N}")

diag = Sigma.diagonal()
print(f"  Annualized vol range: [{np.sqrt(diag.min()*12)*100:.1f}%, {np.sqrt(diag.max()*12)*100:.1f}%]")
print(f"  Median annualized vol: {np.sqrt(np.median(diag)*12)*100:.1f}%")

# NaN impact diagnostic
R_raw = returns_m.loc[eligible, wcols]
n_nan_per_firm = R_raw.isna().sum(axis=1)
firms_with_nan = (n_nan_per_firm > 0).sum()
print(f"\nNaN impact: {firms_with_nan}/{N} firms have missing returns in the window")
if firms_with_nan > 0:
    worst = n_nan_per_firm.nlargest(5)
    print(f"  Worst cases (NaN months / 120):")
    for isin, nn in worst.items():
        print(f"    {firm_names.get(isin, '?')[:40]:<40} {int(nn):>3} NaN ({int(nn)/120*100:.0f}%)")

Year 2013: N = 337 firms, τ = 120 months

Expected return vector μ_hat:
  Shape: (337,)
  Mean:   1.2485% /month (14.98% /year)
  Min:    -0.4916% (CITIGROUP)
  Max:    3.9473% (APPLE)

Covariance matrix Σ:
  Shape: (337, 337)
  Symmetric: True
  PSD: True  (min eigenvalue: -1.05e-16)
  Rank: 119 / 337
  Annualized vol range: [11.9%, 104.2%]
  Median annualized vol: 28.8%

NaN impact: 14/337 firms have missing returns in the window
  Worst cases (NaN months / 120):
    GENERAL MOTORS                            83 NaN (69%)
    DELTA AIR LINES                           40 NaN (33%)
    KBR                                       35 NaN (29%)
    MASTERCARD                                29 NaN (24%)
    SUNPOWER DEAD - DELIST.18/11/24           23 NaN (19%)


The `moments` dictionary is now ready for Section 2.2 (Portfolio Optimization). Each entry contains:
- `mu_hat`: expected return vector $(N,)$
- `Sigma`: covariance matrix $(N \times N)$
- `eligible`: list of ISINs in the investment set
- `window_cols`: estimation window dates

Usage: `mu_hat, Sigma, eligible, wcols = moments[Y]`